# PVT sweep — 8T DCIM bitcell

Orchestrator for the 8T functional/margin tests across process/voltage/temperature corners.

**Runs inside the IIC-OSIC-TOOLS container** (repo mounted at `/workspace`, `ngspice` on PATH).

Division of responsibility (do not blur these):
- **ngspice** = simulation truth. **`params_*.spice`** = sizing truth. **This notebook** = orchestration + tabulation only — no physics is re-implemented here.
- The only file this notebook writes is `designs/pvt.spice` (per corner), and it restores the original at the end.

**Things to review / tune (mine, not signoff):**
- PASS thresholds in `evaluate()` are notebook-side acceptance bands (e.g. 0.9·VDD / 0.1·VDD). Set the real signoff numbers.
- `CORNERS` list — confirm the GF180 lib keywords and the V/T extremes.
- `and_multiply` parsing relies on the deck's 4 trans printing in (0,0),(1,0),(0,1),(1,1) order.

In [ ]:
import subprocess, re
from pathlib import Path
import pandas as pd

# --- paths (inside container; repo mounted at /workspace) ---
ROOT    = Path("/workspace/analog")
PVT     = ROOT / "designs" / "pvt.spice"
SIM8T   = ROOT / "sim" / "8t"
RESULTS = ROOT / "flow" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

GF180 = "/foss/pdks/gf180mcuD/libs.tech/ngspice"

# --- pvt.spice template: the ONLY file rewritten per corner ---
PVT_TEMPLATE = """* File: pvt.spice
* AUTO-GENERATED by pvt_sweep.ipynb -- corner: {name}  (do not hand-edit during a sweep)
.include {gf180}/design.ngspice
.lib {gf180}/sm141064.ngspice {lib}
.temp {temp}
.param VDD_CORNER={vdd}
"""

# --- corners: name, GF180 lib corner, temp (C), VDD (V) ---
CORNERS = [
    dict(name="TT_25_3p3",   lib="tt", temp=25,  vdd=3.30),  # nominal
    dict(name="SS_125_2p97", lib="ss", temp=125, vdd=2.97),  # worst SNM / read-disturb
    dict(name="FF_m40_3p63", lib="ff", temp=-40, vdd=3.63),  # worst write
    dict(name="SS_m40_2p97", lib="ss", temp=-40, vdd=2.97),  # extreme cold-slow
    dict(name="FF_125_3p63", lib="ff", temp=125, vdd=3.63),  # extreme hot-fast
]

# test name -> deck file (all under sim/8t/)
TESTS = ["hold_run", "write_run", "write_margin", "hold_snm",
         "and_multiply", "read_disturb_hi", "read_disturb_lo", "read_run"]

In [ ]:
NUM = r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?"

def write_pvt(corner):
    """Regenerate designs/pvt.spice for one corner."""
    PVT.write_text(PVT_TEMPLATE.format(gf180=GF180, **corner))

def run_ngspice(deck):
    """Headless batch run of one deck; stdout+stderr captured."""
    return subprocess.run(["ngspice", "-b", str(deck)],
                          cwd=str(ROOT), capture_output=True, text=True, timeout=300)

def had_error(proc):
    """ngspice -b often exits 0 even on a bad deck -- also scan output."""
    blob = proc.stdout + proc.stderr
    return proc.returncode != 0 or re.search(r"(?i)(error|undefined parameter|fatal)", blob) is not None

def meas_last(out, name):
    """Last 'name = value' (ngspice meas / print) as float, or None."""
    m = re.findall(rf"^\s*{re.escape(name)}\s*=\s*({NUM})", out, re.M)
    return float(m[-1]) if m else None

def meas_all(out, name):
    """All 'name = value' in order -- for multi-tran decks (and_multiply)."""
    return [float(x) for x in re.findall(rf"^\s*{re.escape(name)}\s*=\s*({NUM})", out, re.M)]

In [ ]:
def evaluate(test, out, vdd):
    """Extract metrics for one deck's stdout. Returns list of {metric, value, pass}.
    PASS bands here are notebook-side acceptance criteria -- tune for signoff."""
    hi, lo = 0.9 * vdd, 0.1 * vdd
    rows = []
    def add(metric, value, ok):
        rows.append({"metric": metric, "value": value, "pass": bool(ok)})

    if test == "hold_run":
        q, qb = meas_last(out, "q_final"), meas_last(out, "qb_final")
        add("q_final",  q,  q  is not None and q  >= hi)
        add("qb_final", qb, qb is not None and qb <= lo)

    elif test == "write_run":            # writes Q: 1 -> 0
        q = meas_last(out, "q_final")
        add("q_final", q, q is not None and q <= lo)

    elif test == "write_margin":         # ngspice computes the margin
        m = re.search(rf"WLWM_RESULT\s*:\s*({NUM})", out)
        margin = float(m.group(1)) if m else None
        endpoints_ok = "[FAIL]" not in out
        add("WLWM_margin", margin, endpoints_ok and margin is not None and margin > 0)

    elif test == "hold_snm":             # ngspice computes the SNM
        snm = meas_last(out, "hold_snm")
        add("hold_snm", snm, snm is not None and snm > 0.15)

    elif test == "and_multiply":         # 4 trans, ordered: (0,0)(1,0)(0,1)(1,1)
        cases = ["(Q0,A0)", "(Q1,A0)", "(Q0,A1)", "(Q1,A1)"]
        mult = meas_all(out, "mult_final")
        for i, c in enumerate(cases):
            v = mult[i] if i < len(mult) else None
            low_expected = (c == "(Q1,A1)")          # product=1 -> RBL discharges (active-low)
            ok = v is not None and (v <= lo if low_expected else v >= hi)
            add(f"rbl{c}", v, ok)

    elif test == "read_disturb_hi":      # Q=1 must hold under repeated reads
        qmin = meas_last(out, "q_min")
        add("q_min", qmin, qmin is not None and qmin > 0.6 * vdd)

    elif test == "read_disturb_lo":      # Q=0 must hold
        qmax = meas_last(out, "q_max")
        add("q_max", qmax, qmax is not None and qmax < 0.4 * vdd)

    elif test == "read_run":             # non-destructive write-port read (this deck: Q=0)
        q, qb = meas_last(out, "q_final"), meas_last(out, "qb_final")
        add("q_final",  q,  q  is not None and q  <= lo)
        add("qb_final", qb, qb is not None and qb >= hi)

    return rows

In [ ]:
# --- the sweep ---
records = []
original_pvt = PVT.read_text()   # restore at the end so git stays clean
try:
    for corner in CORNERS:
        write_pvt(corner)
        for test in TESTS:
            proc = run_ngspice(SIM8T / f"{test}.spice")
            err  = had_error(proc)
            if err:
                print(f"[!] {corner['name']:<13} {test:<16} ngspice flagged an error "
                      f"(rc={proc.returncode}) -- inspect proc output")
            for row in evaluate(test, proc.stdout, corner["vdd"]):
                records.append(dict(corner=corner["name"], lib=corner["lib"],
                                    temp=corner["temp"], vdd=corner["vdd"],
                                    test=test, ran_ok=not err, **row))
finally:
    PVT.write_text(original_pvt)         # always restore, even on exception

df = pd.DataFrame(records)
df.to_csv(RESULTS / "pvt_sweep.csv", index=False)
print(f"{len(df)} rows -> {RESULTS/'pvt_sweep.csv'}")
df

In [ ]:
# --- worst-case report ---
fails = df[~df["pass"]]
print(f"{len(fails)} FAIL row(s)" if len(fails) else "All metrics PASS across all corners.")
if len(fails):
    display(fails[["corner", "test", "metric", "value", "ran_ok"]])

# value of every metric vs corner (quick scan)
df.pivot_table(index=["test", "metric"], columns="corner", values="value", aggfunc="first")

In [ ]:
# --- margin plots ---
import matplotlib.pyplot as plt

margins = df[df["metric"].isin(["hold_snm", "WLWM_margin"])]
piv = margins.pivot_table(index="corner", columns="metric", values="value")
ax = piv.plot.bar(figsize=(8, 4))
ax.set_ylabel("Volts"); ax.set_title("Stability / write margin across PVT corners")
ax.axhline(0, color="k", lw=0.6)
plt.tight_layout(); plt.savefig(RESULTS / "margins.png", dpi=120); plt.show()